---
---
# **Tutorial 2A:** *Tools and Function Calling*
### *Giving the model the ability to do the task !!*
---
---

### QUESTIONS FOR TODAY
> 1.   *An LLM only predicts text. So how can it ever check a live price, query our database, or send an email?*
> 2.   *If the model "calls" our function — who is actually running that code?*

### OUR ROADMAP
| Step | What We Are Building | The Idea |
|---|---|---|
| 1 | **The Problem** | The model has no live data and cannot take actions |
| 2 | **Define the Data + the Tool** | A tool is just a normal Python function |
| 3 | **Describe the Tool** | The *schema*: how the model learns what exists |
| 4 | **Function Calling** | The 4-step handshake, done manually |
| 5 | **Two Tools** | The model *routes* to the right one |
| 6 | **To Call or Not to Call** | The model can also just answer |
| 7 | **The Shortcut** | Let the SDK run the loop for you |

### TASK FOR TODAY
> **ASK → THE MODEL REQUESTS A TOOL → *YOU* RUN IT → SEND THE RESULT BACK → THE MODEL ANSWERS.**

> *This single loop is what turns a text predictor into an agent.*

---
# **Step 0: Setting Up**
---

Same setup as in previous **Tutorials** — we use the current **`google-genai`** SDK and read the API key from Colab's **Secrets**.

> 🔑 **If you already did in previous Tutorials**, your key is ready. If not:
> 1. Get a free key at **[Google AI Studio → API keys](https://aistudio.google.com/apikey)**
> 2. In Colab, click the **🔑 Secrets** icon (left sidebar) → **Add new secret**
> 3. Name: `GOOGLE_API_KEY` → paste your key → turn **Notebook access** ON
>
> ⚠️ *Never paste an API key directly into a code cell. Notebooks get shared and pushed to GitHub.*

In [ ]:
# ---------------------------------------------------------
# 1. INSTALL THE CURRENT SDK
# ---------------------------------------------------------
%pip install -q google-genai

# ---------------------------------------------------------
# 2. BRING IN OUR TOOLKITS
# ---------------------------------------------------------
from google import genai            # the client that talks to Gemini
from google.genai import types      # helper objects (tool definitions, configs)
from google.colab import userdata   # secure access to our stored API key

# ---------------------------------------------------------
# 3. AUTHENTICATE
# ---------------------------------------------------------
client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))

# The 'flash' tier: fast and cheap, perfect for tool calling
MODEL = "gemini-3.1-flash-lite"
#"gemini-3.5-flash-lite"

print("Client ready. Model:", MODEL)

---
# **Step 1: The Problem - *the model cannot take action***
---

Our model is brilliant at language. But it was trained on text that is now frozen in the past, and it **cannot reach out and look anything up**.

Let's prove it. We will ask about a price that lives in *our own company's* system — something no amount of training data could contain.

In [ ]:
# ---------------------------------------------------------
# ASKING ABOUT DATA THE MODEL CANNOT POSSIBLY KNOW
# ---------------------------------------------------------
user_prompt = "What is the flight price from Hyderabad to Kochi in our travel system?"

response = client.models.generate_content(
    model=MODEL,
    contents=user_prompt
)

print(response.text)

Look at what came back. The model either **apologises** that it cannot access live systems, or it **invents a plausible-sounding number**.

Neither is acceptable in a real system. This is the *confidently wrong* problem from Tutorial 2 — and the fix is not a bigger model. The fix is to **give it a tool.**

---

### ❓ Quick Audience Poll ------------------------------------
> Type **A** (needs a tool) or **B** (the model can answer alone):

> **1.**  *"Explain what a REST API is."* <br>
**2.**  *"How many units of product X are in our warehouse right now?"* <br>
**3.**  *"What is 15% of 2,40,000?"* <br>
**4.**  *"What is today's date?"*

---

<details><summary><b>Reveal answers</b> (click)</summary>

**1 = B.** General knowledge, already inside the model. No tool needed.

**2 = A.** Private, live data. Only your database knows this.

**3 = B... mostly.** Modern models handle simple arithmetic, but for anything financial you should still use a tool — a calculator is *exact*, a language model is *probable*. Never let a text predictor be your billing engine.

**4 = A.** Surprising, but true! A model has no clock. It only knows text it was trained on, which stopped at some point in the past. Today's date must be handed to it.
</details>

---
# **Step 2: Define the Data and the Tool**
---

> ### 🔧 **A "tool" is just an ordinary Python function.**

There is no special AI in it. It is the same function you would write on any normal working day.

So let's build our tiny travel system.

First **the data** (in real life this would be a database query or an API call — we use a small dictionary so we can see everything), and then **the function** that reads it.

In [ ]:
# ---------------------------------------------------------
# 1. OUR "DATABASE" (a simple dictionary of routes and prices)
# ---------------------------------------------------------
# The key is a (origin, destination) pair; the value is the price in rupees.
FLIGHT_PRICES = {
    ("Hyderabad", "Kochi"):    7500,
    ("Hyderabad", "Mumbai"):    6000,
    ("Hyderabad", "Bengaluru"): 4750,
    ("Kochi", "Bengaluru"):    2500,
}

# ---------------------------------------------------------
# 2. OUR TOOL (a plain Python function that reads that data)
# ---------------------------------------------------------
def get_flight_price(origin: str, destination: str):
    """Look up the flight price between two cities in our travel system."""

    # .get() returns None if we do not fly that route
    price = FLIGHT_PRICES.get((origin, destination))

    if price is None:
        return "Route not available in our system."

    return f"{price} INR"

In [ ]:
# ---------------------------------------------------------
# TEST THE TOOL LIKE A NORMAL FUNCTION - no AI involved yet!
# ---------------------------------------------------------
print("Hyderabad -> Kochi  :", get_flight_price("Hyderabad", "Kochi"))
print("Hyderabad -> Delhi   :", get_flight_price("Hyderabad", "Delhi"))

It works, and notice: **the AI has not been involved at all so far.** We just wrote and tested a normal function.

That is a genuinely useful mental model — *your tools are ordinary code that you can test, log, and debug on their own*, long before any model touches them.

---
# **Step 3: Describe the Tool to the Model**
---

The model cannot read our Python file. It only reads **text**. So we must hand it a *description* of what the function is called, what it does, and what inputs it needs.

This description is called the **schema** (or *function declaration*). Think of it as the **menu card** we give the model:

| Part of the schema | What it tells the model |
|---|---|
| `name` | **which function** to ask for |
| `description` | **when** it should be used |
| `parameters` | **what information** it must collect from the user |

> ### ⚠️ The `description` is the single most important line you will write.
> It is the only thing the model uses to decide whether to reach for this tool. A vague description gives you a model that ignores your tool or uses it at the wrong moment. **Write it like documentation for a new teammate.**

In [ ]:
# ---------------------------------------------------------
# DECLARING OUR TOOL (the "menu card" for the model)
# ---------------------------------------------------------
flight_tool = types.FunctionDeclaration(
    name="get_flight_price",                      # must match our function name

    description=(                                 # WHEN to use it
        "Get the flight price between two cities from the company travel system. "
        "Use this whenever a user asks about the cost of a flight."
    ),

    parameters={                                  # WHAT inputs it needs
        "type": "OBJECT",
        "properties": {
            "origin":      {"type": "STRING", "description": "The departure city"},
            "destination": {"type": "STRING", "description": "The arrival city"},
        },
        "required": ["origin", "destination"],    # both are compulsory
    },
)

# A "Tool" is simply a container that can hold one or more declarations
travel_tools = types.Tool(function_declarations=[flight_tool])

print("Tool declared:", flight_tool.name)

---
# **Step 4: Function Calling - *the 4-step handshake***
---

Now we connect the two halves. The most common misunderstanding in this entire topic is this:

> ### 🚨 **The model does NOT run your code.**
> It has no access to your machine. It can only **send you a request** that says *"please run `get_flight_price` with these arguments and tell me the answer."*
> **Your code decides whether to obey.** That gap is your security boundary — and it is the reason function calling is safe enough to put in production.

So one question becomes a four-step conversation:

| | Step | Who does it |
|---|---|---|
| 1️⃣ | Send the question **+ the tool menu** | you → model |
| 2️⃣ | Model replies: *"call this tool with these arguments"* | model → you |
| 3️⃣ | **Run the function** | **your code** |
| 4️⃣ | Send the result back; model writes the answer | you → model → you |

Let's walk through all four, one cell at a time.

In [ ]:
# ---------------------------------------------------------
# STEP 1 of 4: SEND THE QUESTION + THE TOOL MENU
# ---------------------------------------------------------
# We start a chat so the conversation remembers itself between turns.
chat = client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(
        tools=[travel_tools],   # <-- here is the menu card

        # We switch OFF the automatic mode on purpose, so we can SEE
        # each step of the handshake ourselves. (Step 7 turns it back on.)
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    ),
)

user_prompt = "I am flying from Hyderabad to Kochi next week for Onam. What will be the flight cost?"

response = chat.send_message(user_prompt)

# Notice: response.text is EMPTY. The model did not answer -
# it asked us to do something first.
print("Model's text answer:", repr(response.text))

In [ ]:
# ---------------------------------------------------------
# STEP 2 of 4: READ THE MODEL'S REQUEST
# ---------------------------------------------------------
# Instead of text, the model sent back a "function call" request.
tool_request = response.function_calls[0]

tool_name = tool_request.name          # which function it wants
tool_args = dict(tool_request.args)    # the arguments it extracted

print("The model is REQUESTING a tool call:")
print("   Tool name :", tool_name)
print("   Arguments :", tool_args)

Pause on that output for a second. 🤯

We wrote *"I am flying from Hyderabad to Kochi next week for Onam"* — plain, messy human language.

The model read it and produced clean, structured arguments: `origin='Hyderabad'`, `destination='Kochi'`.

In [ ]:
# ---------------------------------------------------------
# STEP 3 of 4: **WE** RUN THE FUNCTION
# ---------------------------------------------------------
# A "tool registry" is just a dictionary mapping the tool's name
# to the real Python function. Cleaner than a pile of if/elif,
# and it scales to 50 tools without touching this code.
TOOL_REGISTRY = {
    "get_flight_price": get_flight_price,
}

# Look up the function by name, then call it with the model's arguments.
# The ** unpacks {"origin": "Hyderabad", ...} into origin="Hyderabad", ...
function_to_run = TOOL_REGISTRY[tool_name]
result = function_to_run(**tool_args)

print(f"We ran {tool_name}() ourselves.")
print("Tool result:", result)

In [ ]:
# ---------------------------------------------------------
# STEP 4 of 4: SEND THE RESULT BACK, GET THE FINAL ANSWER
# ---------------------------------------------------------
# We hand the result back in a special "function response" wrapper
# so the model knows this is the answer to its own request.
final_response = chat.send_message(
    types.Part.from_function_response(
        name=tool_name,
        response={"result": result},
    )
)

print("Final answer from the model:\n")
print(final_response.text)

**That is function calling.** The whole idea, complete.

The model supplied the *language understanding* at both ends — reading the messy question, then writing a human sentence around our raw number — and **our code supplied the truth in the middle**. The price in that answer came from our dictionary, not from the model's imagination.

---

### ❓ FAQ ------------------------------------

> *So is the model executing my Python code?*

<details><summary><b>Reveal answer</b> (click)</summary>

**No — and this is the point.** The model only ever sends you a *request* containing a name and some arguments. Your code looks that name up in your registry and decides whether to run it.

This means **you** keep control of everything that matters: permissions, validation, rate limits, logging, and refusing anything suspicious. A tool call is a request to your application, not a command to your machine.
</details>

---
# **Step 5: Two Tools - *the model routes to the right one***
---

One tool is a demo. Real systems have many, and the model must **pick the right one**. Let's add hotels and see it choose.

In [ ]:
# ---------------------------------------------------------
# 1. A SECOND SET OF DATA AND A SECOND TOOL
# ---------------------------------------------------------
HOTEL_PRICES = {
    "Mumbai":     7500,
    "Bengaluru":  5200,
    "Kochi":     2800,
}

def get_hotel_price(city: str):
    """Look up the average hotel price per night in a city."""
    price = HOTEL_PRICES.get(city)

    if price is None:
        return "No hotels listed for this city."

    return f"{price} INR per night"


# ---------------------------------------------------------
# 2. DESCRIBE IT FOR THE MODEL
# ---------------------------------------------------------
hotel_tool = types.FunctionDeclaration(
    name="get_hotel_price",
    description=(
        "Get the average price per night of a hotel in a given city. "
        "Use this whenever a user asks about accommodation or stay costs."
    ),
    parameters={
        "type": "OBJECT",
        "properties": {
            "city": {"type": "STRING", "description": "The city to check hotels in"},
        },
        "required": ["city"],
    },
)

# ---------------------------------------------------------
# 3. BOTH TOOLS ON THE MENU, BOTH IN THE REGISTRY
# ---------------------------------------------------------
travel_tools = types.Tool(function_declarations=[flight_tool, hotel_tool])

TOOL_REGISTRY = {
    "get_flight_price": get_flight_price,
    "get_hotel_price":  get_hotel_price,
}

print("Tools available to the model:", list(TOOL_REGISTRY.keys()))

Now, because we will repeat the same four steps, let's wrap them in **one small helper function**. This is exactly the shape of code you would write in a real application.

In [ ]:
# ---------------------------------------------------------
# THE WHOLE HANDSHAKE, PACKAGED INTO ONE REUSABLE FUNCTION
# ---------------------------------------------------------
def ask(user_prompt):
    """Send a question, handle any tool call, and return the final answer."""

    chat = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            tools=[travel_tools],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        ),
    )

    # --- Step 1: ask ---
    response = chat.send_message(user_prompt)

    # --- If the model did not want a tool, it already answered. ---
    if not response.function_calls:
        print("(No tool needed)")
        return response.text

    # --- Step 2: read the request ---
    tool_request = response.function_calls[0]
    tool_name = tool_request.name
    tool_args = dict(tool_request.args)
    print(f"Model requested: {tool_name}({tool_args})")

    # --- Step 3: we run it ---
    result = TOOL_REGISTRY[tool_name](**tool_args)
    print(f"Tool returned : {result}")

    # --- Step 4: send the result back ---
    final = chat.send_message(
        types.Part.from_function_response(
            name=tool_name,
            response={"result": result},
        )
    )
    return final.text

In [ ]:
# ---------------------------------------------------------
# TEST 1: a question about FLIGHTS
# ---------------------------------------------------------
#print(ask("How much is a flight from Hyderabad to Bengaluru?"))
print(ask("How much is a flight from HYD to Kochi?"))

In [ ]:
# ---------------------------------------------------------
# TEST 2: a question about HOTELS (same code, different tool chosen!)
# ---------------------------------------------------------
print(ask("What do hotels cost in Mumbai?"))

We changed nothing but the question, and the model **routed to a different tool by itself** — purely from the `description` we wrote. This is why the descriptions matter so much.

> 🧪 **Ask** *"How much is a flight to Chennai from Hyderabad?"* — a route missing from our data.

In [ ]:
# Try it !
print(ask("How much is a flight to Chennai from Hyderabad?"))

---
# **Step 6: To Call or Not to Call**
---

Tools are an **option** we offer, not an obligation. If a question needs no tool, the model should simply answer.

Our `ask()` helper already handles this — remember the `if not response.function_calls:` check. Let's watch it work.

In [ ]:
# ---------------------------------------------------------
# A GENERAL QUESTION - no tool should be used here
# ---------------------------------------------------------
print(ask("Tell me about yourself."))

No tool call. The model recognised that our travel tools were irrelevant and just replied.

**This deciding behaviour is what makes it an agent** rather than a script. In Tutorial 1 we wrote a chatbot where *we* used `if/else` to decide what happens next. Here, **the model decides**, and our code carries out the decision. That inversion is the entire idea behind agentic systems.

---
# **Step 7: The Shortcut - *let the SDK run the loop***
---

Now that you know exactly what the four steps are, here is the good news: **the SDK can do all of them for you.**

Pass your **plain Python functions** straight into `tools=[...]` and remove the "disable" line. The SDK will read your function's *name, type hints and docstring* to build the schema automatically, then run the whole handshake behind the scenes.

> This is why the docstring and type hints on your tool functions are not decoration — in this mode, **they become the schema the model reads.**

In [ ]:
# ---------------------------------------------------------
# AUTOMATIC FUNCTION CALLING - the whole handshake in one call
# ---------------------------------------------------------
auto_chat = client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(
        # Pass the FUNCTIONS themselves, not the declarations.
        # No 'automatic_function_calling=disable' line, so it stays ON.
        tools=[get_flight_price, get_hotel_price],
    ),
)

response = auto_chat.send_message(
    "I am flying Hyderabad to Kochi and staying in Kochi. What are both costs?"
)

print(response.text)

One call, and the model used **both** tools to build a complete answer.

So when should you use each style?

| | **Manual** (Steps 1–6) | **Automatic** (Step 7) |
|---|---|---|
| **You control** | every step | the SDK handles it |
| **Good for** | validation, permissions, logging, approvals | quick prototypes, simple tools |
| **In production** | usually this — because a tool that spends money or writes to a database needs a checkpoint | fine for read-only, harmless tools |

> 💡 **Architect's note:** the manual loop is not the "beginner" version — it is the version where you can insert a human approval, check a permission, or log the call. Convenience is lovely **until a tool books a flight nobody asked for.**

---
# **What We Built Today**
---

We started with a model that could not tell us a single real price. We finished with one that reads a messy sentence, picks the right function, gets real data from our own system, and explains it in a human answer.

And the loop behind all of it was just:

> **ASK → THE MODEL REQUESTS A TOOL → *YOU* RUN IT → SEND THE RESULT BACK → THE MODEL ANSWERS.**

Three ideas worth carrying forward:

1. **A tool is just a function.** Ordinary code you can test and debug on its own.
2. **The `description` is the interface.** The model chooses tools by reading your words. Vague words, wrong choices.
3. **The model requests; your code executes.** That gap is where your validation, permissions and logging live — and it is what makes this safe to ship.

Our tools today only *read* data. The moment a tool can *write* — send the email, book the ticket, update the record — the same loop becomes a system that takes real action in the world. That is where we are heading next.

---
# **Thank you !**
---

Author: *Aneetta Sara Shany*

Date: *2026 August 01, Saturday*